### Ячейка 1: Импорты и Основные Настройки

In [13]:
# -*- coding: utf-8 -*-
# Ячейка 1: Импорты и Основные Настройки (v2.0 - Чистая версия для Ансамбля)

# Стандартные библиотеки Python
import os
import gc
import sys
import json
import re
import warnings
import math
import traceback
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter, defaultdict, OrderedDict # Для ансамбля и анализа

# Подавление стандартных предупреждений (опционально)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Основные библиотеки Data Science
import numpy as np
import pandas as pd
import Levenshtein # pip install python-Levenshtein

# PyTorch и связанные библиотеки
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split # random_split для val
from torch.nn.utils.rnn import pad_sequence
import torchaudio # Может понадобиться для SpecAugment в MorseDataset

# Утилиты и Визуализация
from tqdm.notebook import tqdm
# Убираем matplotlib/seaborn, если не делаем детальный анализ ошибок с графиками

# --- Определение Корня Проекта (ВАЖНО!) ---
# Эта логика должна быть такой же, как в основном ноутбуке
PROJECT_ROOT: Optional[Path] = None
try:
    current_working_dir = Path(os.getcwd()).resolve()
    potential_root = None
    if (current_working_dir / 'src').is_dir(): potential_root = current_working_dir
    elif (current_working_dir.parent / 'src').is_dir(): potential_root = current_working_dir.parent
    else:
        temp_path = current_working_dir
        for _ in range(3):
            if (temp_path / 'src').is_dir(): potential_root = temp_path; break
            if temp_path == temp_path.parent: break
            temp_path = temp_path.parent
    if potential_root and (potential_root / 'src').is_dir(): PROJECT_ROOT = potential_root
    else: PROJECT_ROOT = current_working_dir.parent # Запасной вариант
    print(f"Предполагаемый корень проекта: {PROJECT_ROOT}")
    project_root_str = str(PROJECT_ROOT)
    if project_root_str not in sys.path: sys.path.insert(0, project_root_str)
except Exception as e_proj_root: print(f"Ошибка определения корня проекта: {e_proj_root}"); raise

# --- Импорты из src ---
try:
    # Модель и блоки
    from src.models.architectures import MorseRecognizer
    # Данные и обработка
    from src.data_processing.datasets import MorseDataset, collate_fn
    from src.features.spectral import get_features # Нужен для MorseDataset
    # Текст и утилиты
    from src.data_processing.text import create_char_map
    from src.utils.path_utils import create_full_path
    # Декодер
    from src.inference.predict import ctc_greedy_decode # Используем декодер из src
except ImportError as e_import_src:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Не удалось импортировать модуль из 'src': {e_import_src}")
    print("   Убедитесь, что корень проекта определен верно и структура папок корректна.")
    raise

# --- Основные Настройки ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {DEVICE}")

# Очистка памяти GPU (на всякий случай)
if DEVICE == torch.device('cuda'):
    torch.cuda.empty_cache()
    gc.collect()

import inspect
try:
    datasets_module = inspect.getmodule(MorseDataset)
    if datasets_module and hasattr(datasets_module, '__file__'):
        print(f"\n[ПРОВЕРКА ИМПОРТА] MorseDataset импортирован из файла: {datasets_module.__file__}")
    else:
        print("\n[ПРОВЕРКА ИМПОРТА] Не удалось определить файл для MorseDataset.")
except Exception as e_inspect:
    print(f"\n[ПРОВЕРКА ИМПОРТА] Ошибка при проверке пути импорта: {e_inspect}")
print("\nЯчейка 1 (Импорты и Настройки) выполнена.")

Предполагаемый корень проекта: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder
Используемое устройство: cuda

[ПРОВЕРКА ИМПОРТА] MorseDataset импортирован из файла: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\src\data_processing\datasets.py

Ячейка 1 (Импорты и Настройки) выполнена.


### Ячейка 2: Определение Классов Модели, Dataset и Collate Fn


In [14]:
# Ячейка 2: Конфигурация Ансамбля
# -------------------------------

# --- 1. Параметры Данных и CTC (Должны быть согласованы с обучением) ---
# Используем PROJECT_ROOT для путей
DATA_DIR = PROJECT_ROOT / "data" / "raw" # Путь к папке с train.csv, sample_submission.csv
print(DATA_DIR)
TRAIN_CSV_FILENAME = "train.csv"
TEST_CSV_FILENAME = "sample_submission.csv"

MORSE_CODE_COLUMN = 'message' # Имя колонки с текстом Морзе
FILE_ID_COLUMN = 'id'         # Имя колонки с ID файла

RANDOM_SEED = 42              # Сид для воспроизводимости разделения train/val (если анализ на val нужен)
VAL_SPLIT_RATIO = 0.1         # Доля данных для валидации

# Параметры CTC (важно для декодирования и collate_fn)
BLANK_CHAR = "<blank>"
PAD_CHAR = "<pad>" # Используется как padding_value в collate_fn
BLANK_IDX = 0
PAD_IDX = 0 # Используем BLANK_IDX для паддинга таргетов в collate_fn

# --- 2. Определение Моделей для Ансамбля ---
# Укажите пути к файлам моделей (.pth) и их конфигураций (.json)
# Пути могут быть относительными от PROJECT_ROOT или абсолютными.
MODELS_TO_ANALYZE = [
    {
        'name': 'Model_0.2607', # Дайте модели осмысленное имя
        'model_path': 'UnionSubmissions/0.2607/MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_193716_best_epoch3_lev0.2607.pth',
        'config_path': 'UnionSubmissions/0.2607/config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json'
    },
    {
        'name': 'Model_0.2637',
        'model_path': 'UnionSubmissions/0.2637/MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_033740_best_epoch23_lev0.2637.pth',
        'config_path': 'UnionSubmissions/0.2637/config_final_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json'
    },
    {
        'name': 'Model_0.2638',
        'model_path': 'UnionSubmissions/0.2638/MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_171419_best_epoch2_lev0.2637.pth', # Lev в имени файла может отличаться от папки
        'config_path': 'UnionSubmissions/0.2638/config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json'
    },
    {
        'name': 'Model_0.2657',
        'model_path': 'UnionSubmissions/0.2657/MorseCRNN_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean_FINETUNE_OneCycleLR_20250421_033740_best_epoch13_lev0.2657.pth',
        'config_path': 'UnionSubmissions/0.2657/config_final_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json'
    },
    {
        'name': 'Model_0.2773',
        'model_path': 'UnionSubmissions/0.2773/morse_v9.1_FT_K3x5_CRNN_ResNetSE_K3x5-K3x5_Hop96_v9.1_FINETUNE_FINETUNE_OneCycleLR_20250420_210451_best_epoch9_lev0.2773.pth',
        'config_path': 'UnionSubmissions/0.2773/config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin.json'
    },
    # Добавьте сюда другие модели при необходимости
]

# --- 3. Настройки Вывода ---
OUTPUT_ENSEMBLE_DIR = PROJECT_ROOT / "UnionSubmissions" / "ensemble_output_v2" # Новая папка для вывода
SUBMISSION_FILENAME = "submission_ensemble_majority.csv"

# --- 4. Настройки Анализа/Извлечения (Опционально) ---
RUN_VALIDATION_ANALYSIS = False # Поставить True для анализа ошибок на валидации
EXTRACT_SECRET_MESSAGE = True  # Поставить True для извлечения сообщения
N_LAST_FILES_FOR_MESSAGE = 17
ARTIFACT_CHARS_TO_REMOVE_SUFFIX = '0Ъ' # Символы для удаления с КОНЦА каждой части

# Проверка существования директории с данными
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Директория с данными не найдена: {DATA_DIR.resolve()}")

print("Ячейка 2 (Конфигурация Ансамбля) выполнена.")
print(f"  Количество моделей для анализа: {len(MODELS_TO_ANALYZE)}")
print(f"  Анализ на валидации: {'Включен' if RUN_VALIDATION_ANALYSIS else 'Выключен'}")
print(f"  Извлечение сообщения: {'Включено' if EXTRACT_SECRET_MESSAGE else 'Выключено'}")
print(f"  Папка для вывода: {OUTPUT_ENSEMBLE_DIR.resolve()}")

C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\data\raw
Ячейка 2 (Конфигурация Ансамбля) выполнена.
  Количество моделей для анализа: 5
  Анализ на валидации: Выключен
  Извлечение сообщения: Включено
  Папка для вывода: C:\Users\vasja\OneDrive\Рабочий стол\MorseAudioDecoder\UnionSubmissions\ensemble_output_v2


### Ячейка 3: Конфигурация Анализа и Загрузка Данных/Моделей

In [15]:
# Ячейка 3: Загрузка Данных и Создание Словарей
# --------------------------------------------

# Глобальные переменные для данных
train_df_full: Optional[pd.DataFrame] = None
test_df: Optional[pd.DataFrame] = None
train_split_df: Optional[pd.DataFrame] = None
val_split_df: Optional[pd.DataFrame] = None
char_to_int: Dict[str, int] = {}
int_to_char: Dict[int, str] = {}
vocab_size: int = 0

print("\n--- Загрузка данных и создание словарей ---")
try:
    # --- Загрузка CSV ---
    train_csv_path = DATA_DIR / TRAIN_CSV_FILENAME
    test_csv_path = DATA_DIR / TEST_CSV_FILENAME
    if not train_csv_path.is_file(): raise FileNotFoundError(f"{TRAIN_CSV_FILENAME} не найден в {DATA_DIR}")
    if not test_csv_path.is_file(): raise FileNotFoundError(f"{TEST_CSV_FILENAME} не найден в {DATA_DIR}")

    train_df_full = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    print(f"Загружен {TRAIN_CSV_FILENAME}: {len(train_df_full)} строк")
    print(f"Загружен {TEST_CSV_FILENAME}: {len(test_df)} строк")

    # Проверка колонок
    if MORSE_CODE_COLUMN not in train_df_full.columns: raise ValueError(f"Колонка '{MORSE_CODE_COLUMN}' не найдена в {TRAIN_CSV_FILENAME}")
    if FILE_ID_COLUMN not in train_df_full.columns: raise ValueError(f"Колонка '{FILE_ID_COLUMN}' не найдена в {TRAIN_CSV_FILENAME}")
    if FILE_ID_COLUMN not in test_df.columns: raise ValueError(f"Колонка '{FILE_ID_COLUMN}' не найдена в {TEST_CSV_FILENAME}")

    # --- Создание словарей ---
    # Используем импортированную функцию create_char_map
    ctc_config_for_map = {'blank_char': BLANK_CHAR, 'pad_char': PAD_CHAR, 'blank_idx': BLANK_IDX, 'pad_idx': PAD_IDX}
    char_to_int, int_to_char, vocab_size = create_char_map(
        train_df_full[MORSE_CODE_COLUMN].astype(str).tolist(), ctc_config_for_map
    )
    print(f"Словарь создан: {vocab_size} символов (включая бланк '{BLANK_CHAR}'={BLANK_IDX}).")

    # --- Разделение на Train/Validation (только если RUN_VALIDATION_ANALYSIS == True) ---
    if RUN_VALIDATION_ANALYSIS:
        print(f"\nРазделение данных для валидации (Ratio={VAL_SPLIT_RATIO}, Seed={RANDOM_SEED})...")
        if not (0 < VAL_SPLIT_RATIO < 1): raise ValueError("Некорректный VAL_SPLIT_RATIO.")
        val_size = int(len(train_df_full) * VAL_SPLIT_RATIO)
        train_size = len(train_df_full) - val_size
        if val_size <= 0 or train_size <= 0: raise ValueError("Некорректные размеры train/val.")

        generator = torch.Generator().manual_seed(RANDOM_SEED)
        train_subset, val_subset = random_split(range(len(train_df_full)), [train_size, val_size], generator=generator)
        train_split_df = train_df_full.iloc[train_subset.indices].copy().reset_index(drop=True)
        val_split_df = train_df_full.iloc[val_subset.indices].copy().reset_index(drop=True)
        print(f"Данные разделены: Train={len(train_split_df)}, Val={len(val_split_df)}")
    else:
        print("\nРазделение на Train/Validation пропущено (RUN_VALIDATION_ANALYSIS=False).")

except FileNotFoundError as e: print(f"❌ Ошибка: {e}"); raise
except ValueError as e: print(f"❌ Ошибка: {e}"); raise
except Exception as e: print(f"❌ Неизвестная ошибка: {e}"); traceback.print_exc(limit=1); raise

print("\nЯчейка 3 (Загрузка Данных) выполнена.")


--- Загрузка данных и создание словарей ---
Загружен train.csv: 30000 строк
Загружен sample_submission.csv: 5000 строк
Найдено уникальных символов в текстах (44):  #0123456789АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ
Размер словаря (Vocab Size, включая бланк): 45
Словарь (idx: char): {0: '<blank>', 1: ' ', 2: '#', 3: '0', 4: '1', 5: '2', 6: '3', 7: '4', 8: '5', 9: '6', 10: '7', 11: '8', 12: '9', 13: 'А', 14: 'Б', 15: 'В', 16: 'Г', 17: 'Д', 18: 'Е', 19: 'Ж', 20: 'З', 21: 'И', 22: 'Й', 23: 'К', 24: 'Л', 25: 'М', 26: 'Н', 27: 'О', 28: 'П', 29: 'Р', 30: 'С', 31: 'Т', 32: 'У', 33: 'Ф', 34: 'Х', 35: 'Ц', 36: 'Ч', 37: 'Ш', 38: 'Щ', 39: 'Ъ', 40: 'Ы', 41: 'Ь', 42: 'Э', 43: 'Ю', 44: 'Я'}
Индекс Blank: 0, Индекс Pad (для collate_fn): 0
Словарь создан: 45 символов (включая бланк '<blank>'=0).

Разделение на Train/Validation пропущено (RUN_VALIDATION_ANALYSIS=False).

Ячейка 3 (Загрузка Данных) выполнена.


### Ячейка 4: Функции Инференса и Анализа Ошибок

In [16]:
# Ячейка 4: Загрузка Моделей (ИСПРАВЛЕНИЕ: Принудительное задание ПРАВИЛЬНОГО пути)
# ------------------------------------------------------------------------------

import re
from collections import OrderedDict
import numpy as np
import traceback
from pathlib import Path # Убедимся, что Path импортирован
import json # Убедимся, что json импортирован
import torch # Нужен для Optional[nn.Module], torch.device, torch.load
import torch.nn as nn # Нужен для Optional[nn.Module]

# --- Убедимся, что глобальные переменные доступны ---
# Проверки на существование PROJECT_ROOT, DATA_DIR, MORSE_CODE_COLUMN, etc. должны быть в Ячейке 2 или 3
# Проверка vocab_size будет ниже перед циклом

def extract_lev_from_filename(filename: str) -> float:
    """Извлекает Levenshtein из имени файла формата ..._levX.XXXX.pth"""
    match = re.search(r"lev([\d.]+)(?:\.pth)?$", filename)
    if match:
        try: return float(match.group(1))
        except ValueError: return float('inf')
    return float('inf')

def load_model_and_config(model_info: Dict[str, str], device: torch.device, global_vocab_size: int) -> Tuple[Optional[nn.Module], Optional[Dict]]:
    """
    Загружает модель и ее конфиг.
    ВАЖНО: Принудительно устанавливает правильный audio_folder_name,
    игнорируя значение из загруженного .json файла.
    """
    try:
        model_path_rel = model_info['model_path']
        config_path_rel = model_info['config_path']
        model_name = model_info['name']
    except KeyError as e:
        print(f"❌ Ошибка: Отсутствует ключ '{e}' в model_info: {model_info}")
        return None, None

    # Абсолютные пути к модели и конфигу
    model_path = (PROJECT_ROOT / model_path_rel).resolve()
    config_path = (PROJECT_ROOT / config_path_rel).resolve()

    if not model_path.is_file(): print(f"Ошибка: Файл модели не найден для '{model_name}': {model_path}"); return None, None
    if not config_path.is_file(): print(f"Ошибка: Файл конфигурации не найден для '{model_name}': {config_path}"); return None, None

    print(f"\nЗагрузка модели '{model_name}'...")
    try:
        # 1. Загрузка конфига из файла
        with open(config_path, 'r', encoding='utf-8') as f: config = json.load(f)
        print(f"  Конфиг '{config_path.name}' загружен (пути в нем будут проигнорированы/перезаписаны).")

        # 2. Обновление/Добавление необходимых параметров в ЗАГРУЖЕННЫЙ config
        if 'model' not in config: config['model'] = {}
        config['model']['vocab_size'] = global_vocab_size # Используем глобальный vocab_size
        # Вычисляем freq_dim, если его нет
        if 'freq_dim' not in config['model']:
            n_fft = config.get("audio", {}).get("n_fft")
            if n_fft and isinstance(n_fft, int) and n_fft > 0:
                config['model']['freq_dim'] = n_fft // 2 + 1
                print(f"  Вычислен freq_dim: {config['model']['freq_dim']}")
            else:
                # Пытаемся взять n_fft из глобального конфига, если в локальном нет
                n_fft_global = CONFIG_GLOBAL.get("audio", {}).get("n_fft") if 'CONFIG_GLOBAL' in globals() else None # CONFIG_GLOBAL - это наш CONFIG из Ячейки 3 MorseAudioDecoder, надо его как-то получить или задать n_fft явно
                n_fft_to_use = n_fft or n_fft_global or 512 # Запасной вариант - 512
                print(f"Предупреждение: n_fft не найден в конфиге {config_path.name}. Используется {n_fft_to_use}.")
                config['model']['freq_dim'] = n_fft_to_use // 2 + 1


        # --- ПРИНУДИТЕЛЬНАЯ УСТАНОВКА ПРАВИЛЬНОГО ПУТИ К АУДИО ---
        # Мы ЗНАЕМ правильный относительный путь в текущем проекте
        CORRECT_RELATIVE_AUDIO_PATH = "data/raw/morse_dataset/morse_dataset" # <-- УКАЖИ ЗДЕСЬ ПРАВИЛЬНЫЙ ПУТЬ ОТ КОРНЯ ПРОЕКТА
        print(f"  ---> УСТАНОВКА КОРРЕКТНОГО ПУТИ: Устанавливаем paths.audio_folder_name = '{CORRECT_RELATIVE_AUDIO_PATH}'")
        if 'paths' not in config: config['paths'] = {}
        config['paths']['audio_folder_name'] = CORRECT_RELATIVE_AUDIO_PATH
        # --- КОНЕЦ ПРИНУДИТЕЛЬНОЙ УСТАНОВКИ ---

        # Добавляем остальные параметры, которые могут понадобиться MorseDataset
        config['morse_code_column'] = MORSE_CODE_COLUMN # Из Ячейки 2
        config['test_file_column'] = FILE_ID_COLUMN    # Из Ячейки 2
        config['train_file_column'] = FILE_ID_COLUMN   # Из Ячейки 2
        if 'ctc' not in config: config['ctc'] = {}
        config['ctc']['pad_idx'] = PAD_IDX             # Из Ячейки 2
        config['ctc']['blank_idx'] = BLANK_IDX           # Из Ячейки 2
        # Добавляем audio config, если его не было, из глобальных настроек (если они есть)
        if 'audio' not in config:
            config['audio'] = CONFIG_GLOBAL.get('audio', {"sample_rate": 8000, "n_fft": 512, "hop_length": 96}) # Примерные значения
            print(f"Предупреждение: Секция 'audio' не найдена в конфиге {config_path.name}. Используются значения: {config['audio']}")

        # 3. Инициализация модели
        model = MorseRecognizer(config).to(device)

        # 4. Загрузка весов
        checkpoint = torch.load(model_path, map_location=device)
        state_dict_to_load = None
        if isinstance(checkpoint, dict):
            keys_found = [k for k in ['model_state_dict', 'state_dict'] if k in checkpoint]
            if keys_found: state_dict_to_load = checkpoint[keys_found[0]]
        if state_dict_to_load is None: state_dict_to_load = checkpoint

        new_state_dict = OrderedDict()
        has_module_prefix = any(k.startswith('module.') for k in state_dict_to_load.keys())
        for k, v in state_dict_to_load.items():
            name = k[7:] if has_module_prefix and k.startswith('module.') else k
            new_state_dict[name] = v

        model.load_state_dict(new_state_dict)
        model.eval()
        print(f"  Модель '{model_name}' успешно загружена и переведена в режим eval().")
        return model, config

    except ValueError as ve: print(f"❌ Ошибка конфигурации для '{model_name}': {ve}"); return None, None
    except FileNotFoundError as fe: print(f"❌ Ошибка файла для '{model_name}': {fe}"); return None, None
    except Exception as e: print(f"❌ Ошибка при загрузке модели '{model_name}': {e}"); traceback.print_exc(limit=1); return None, None

# --- Загрузка всех моделей из списка ---
loaded_models_data: Dict[str, Dict] = {}
print("\n" + "="*30 + " Загрузка Моделей для Ансамбля " + "="*30)

if not MODELS_TO_ANALYZE:
    print("Список MODELS_TO_ANALYZE пуст. Модели не загружены.")
else:
    # Проверка наличия vocab_size перед циклом
    if 'vocab_size' not in locals() or not vocab_size:
         raise RuntimeError("Переменная vocab_size не определена или равна 0. Выполните Ячейку 3.")

    for model_info in MODELS_TO_ANALYZE:
        model, config = load_model_and_config(model_info, DEVICE, vocab_size)
        if model and config:
            base_lev = extract_lev_from_filename(model_info['model_path'])
            loaded_models_data[model_info['name']] = {
                'model': model,
                'config': config, # Сохраняем ИСПРАВЛЕННЫЙ конфиг
                'base_lev': base_lev
            }
            print(f"  Levenshtein из имени файла: {f'{base_lev:.4f}' if np.isfinite(base_lev) else 'N/A'}")

    if not loaded_models_data:
        print("\n⚠️ Не удалось загрузить ни одной модели. Проверьте пути и ошибки выше.")
    else:
        print(f"\nУспешно загружено моделей: {len(loaded_models_data)}")
        print(f"  Имена загруженных моделей: {list(loaded_models_data.keys())}")

print("\nЯчейка 4 (Загрузка Моделей) выполнена.")


============================== Загрузка Моделей для Ансамбля ==============================

Загрузка модели 'Model_0.2607'...
  Конфиг 'config_initial_CRNN_ResNetSE_K3x5-K3x5_Hop96_v10.0_FINETUNE_GenericPerlin_Clean.json' загружен (пути в нем будут проигнорированы/перезаписаны).
  ---> УСТАНОВКА КОРРЕКТНОГО ПУТИ: Устанавливаем paths.audio_folder_name = 'data/raw/morse_dataset/morse_dataset'

--- Инициализация ResNet-SE CNN слоев (из architectures.py) ---
  Начальная F=257, SE=True (ratio=16)
  Block 1: ResBlock(1, 32, k=(3, 5), s=(2, 2), se=True)
  Block 2: ResBlock(32, 64, k=(3, 5), s=(2, 2), se=True)
CNN: Общий фактор сжатия: Время=4, Частота=4
--- Инициализация RNN ---
  BiGRU: Input=4160, Hidden=256, Layers=2, Dropout=0.20
--- Инициализация Классификатора ---
  Dropout: 0.20
  Linear: In=512, Out=45
----------------------------------------
  Модель 'Model_0.2607' успешно загружена и переведена в режим eval().
  Levenshtein из имени файла: 0.2607

Загрузка модели 'Model_0.2637'...

### Ячейка 5: Запуск Анализа Ошибок на Валидации

In [17]:
# Ячейка 5: (Опционально) Анализ Ошибок на Валидации
# -------------------------------------------------

all_val_error_stats: Dict[str, Dict] = {}
all_val_levenshtein: Dict[str, float] = {}

if RUN_VALIDATION_ANALYSIS:
    print("\n" + "="*30 + " Анализ Ошибок Моделей на Валидации " + "="*30)

    if val_split_df is None or val_split_df.empty:
        print("Ошибка: Валидационный DataFrame (val_split_df) не создан. Пропустите этот шаг или включите разделение в Ячейке 3.")
    elif not loaded_models_data:
        print("Ошибка: Нет загруженных моделей для анализа.")
    else:
        # --- Функция для получения предсказаний (адаптированная) ---
        def get_predictions_val(
            model_name: str, loaded_data: Dict[str, Dict], df_val: pd.DataFrame,
            char2int: Dict[str, int], int2char: Dict[int, str], device: torch.device
        ) -> Tuple[List[str], List[str], float]:
            """Получает предсказания и GT для валидации, вычисляет Levenshtein."""
            model_data = loaded_data[model_name]
            model = model_data['model']; config = model_data['config']
            pad_idx = config.get("ctc", {}).get("pad_idx", -1)
            blank_idx = config.get("ctc", {}).get("blank_idx", 0)
            batch_size = config.get("training", {}).get("batch_size", 16) * 2
            num_workers = config.get("num_workers", 0)

            try:
                dataset = MorseDataset(
                    df=df_val,
                    char_to_int=char2int,
                    config=config,
                    is_train=False,
                    audio_augmenter=None,
                    spec_augment_transform=None,
                    project_root=PROJECT_ROOT, # <<< ЯВНО ПЕРЕДАЕМ PROJECT_ROOT
                    apply_all_augmentations_flag=False
                )
                # -------------------------------------------
                collate_wrapper = lambda batch: collate_fn(batch, pad_idx)
                dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_wrapper, pin_memory=(device.type == 'cuda'))
            except Exception as e_dl: print(f"Ошибка DataLoader для '{model_name}': {e_dl}"); return [], [], float('inf')

            preds_list = []
            gt_list = []
            total_lev_dist = 0
            total_len_gt = 0

            pbar = tqdm(dataloader, desc=f"Валидация '{model_name}'", leave=False)
            with torch.no_grad():
                for batch_data in pbar:
                    if batch_data is None: continue
                    try:
                        features, targets, _, target_lengths = batch_data
                        features = features.to(device)
                        logits = model(features)
                        decoded_batch = ctc_greedy_decode(logits.detach(), int2char, blank_idx)
                        preds_list.extend(decoded_batch)

                        targets_cpu = targets.cpu(); target_lengths_cpu = target_lengths.cpu().numpy()
                        for i in range(targets_cpu.size(0)):
                            length = target_lengths_cpu[i]
                            if length > 0 and length <= targets_cpu.shape[1]:
                                target_indices = targets_cpu[i, :length].numpy()
                                gt_text = "".join([int2char.get(idx, '?') for idx in target_indices if idx != pad_idx and idx != blank_idx])
                                gt_list.append(gt_text)
                                total_lev_dist += Levenshtein.distance(decoded_batch[i], gt_text)
                                total_len_gt += len(gt_text) # Считаем общую длину GT для CER
                            else: gt_list.append("") # Добавляем пустую строку, если длина 0
                    except Exception as e_batch_val:
                        print(f"Ошибка в батче валидации для '{model_name}': {e_batch_val}")
                        preds_list.extend(["ERROR"] * batch_data[0].shape[0])
                        gt_list.extend(["ERROR"] * batch_data[0].shape[0])

            avg_lev = (total_lev_dist / len(gt_list)) if gt_list else float('inf')
            return preds_list, gt_list, avg_lev

        # --- Цикл анализа по моделям ---
        for model_name in loaded_models_data.keys():
            print(f"\n--- Анализ на валидации для: '{model_name}' ---")
            val_preds, val_gt, avg_lev = get_predictions_val(
                model_name, loaded_models_data, val_split_df, char_to_int, int_to_char, DEVICE
            )
            all_val_levenshtein[model_name] = avg_lev
            print(f"  Среднее расстояние Левенштейна (Val): {avg_lev:.4f}")

            # Опционально: Краткий анализ ошибок (без графиков)
            if val_preds and val_gt and len(val_preds) == len(val_gt):
                # Используем функцию из старой ячейки 4 (нужно ее определить или импортировать)
                # Для краткости просто выведем Levenshtein
                pass # Можно добавить вызов analyze_character_errors и print_top_errors сюда
            else:
                print("  Пропуск детального анализа ошибок из-за проблем с предсказаниями/GT.")

            # Очистка
            del val_preds, val_gt
            gc.collect()
            if DEVICE == torch.device('cuda'): torch.cuda.empty_cache()

        print("\nАнализ на валидации завершен.")
else:
    print("\nАнализ ошибок на валидации пропущен (RUN_VALIDATION_ANALYSIS=False).")

print("\nЯчейка 5 (Анализ на Валидации) выполнена.")


Анализ ошибок на валидации пропущен (RUN_VALIDATION_ANALYSIS=False).

Ячейка 5 (Анализ на Валидации) выполнена.


### Ячейка 6: Генерация Предсказаний на Тестовом Наборе

In [18]:
# Ячейка 6: Инференс на Тесте (ИСПРАВЛЕНА ОШИБКА ОТСТУПОВ)
# --------------------------------------------------------

all_test_predictions: Dict[str, List[str]] = {}

print("\n" + "="*30 + " Генерация Предсказаний на Тестовом Наборе " + "="*30)

# --- НАЧАЛО БЛОКА С ПРАВИЛЬНЫМ ОТСТУПОМ ---
if test_df is None or test_df.empty:
    print("Ошибка: Тестовый DataFrame (test_df) не загружен. Выполните Ячейку 3.")
elif not loaded_models_data:
    print("Ошибка: Нет загруженных моделей для инференса.")
else:
    # --- Функция для получения предсказаний на тесте ---
    # (Определение функции get_predictions_test остается здесь, с отступом)
    def get_predictions_test(
        model_name: str, loaded_data: Dict[str, Dict], df_test: pd.DataFrame,
        char2int: Dict[str, int], int2char: Dict[int, str], device: torch.device
    ) -> Optional[List[str]]:
        # ... (код функции get_predictions_test) ...
        model_data = loaded_data[model_name]
        model = model_data['model']; config = model_data['config']
        pad_idx = config.get("ctc", {}).get("pad_idx", -1)
        blank_idx = config.get("ctc", {}).get("blank_idx", 0)
        batch_size = config.get("training", {}).get("batch_size", 16) * 2
        num_workers = config.get("num_workers", 0)
        try:
            dataset = MorseDataset(
                df=df_test,
                char_to_int=char2int,
                config=config,
                is_train=False,
                audio_augmenter=None,
                spec_augment_transform=None,
                project_root=PROJECT_ROOT,
                apply_all_augmentations_flag=False
            )
            collate_wrapper = lambda batch: collate_fn(batch, pad_idx)
            dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_wrapper, pin_memory=(device.type == 'cuda'))
        except Exception as e_dl: print(f"Ошибка DataLoader для '{model_name}': {e_dl}"); return None

        preds_list = []
        pbar = tqdm(dataloader, desc=f"Инференс (Тест) '{model_name}'", leave=False)
        with torch.no_grad():
            for batch_data in pbar:
                if batch_data is None: continue
                try:
                    features, _, _, _ = batch_data
                    features = features.to(device)
                    logits = model(features)
                    if not torch.isfinite(logits).all():
                         print(f"Предупреждение (Тест {model_name}): NaN/Inf в логитах!")
                         decoded_batch = ["ERROR_LOGITS"] * features.shape[0]
                    else:
                         decoded_batch = ctc_greedy_decode(logits.detach(), int2char, blank_idx)
                    preds_list.extend(decoded_batch)
                except Exception as e_batch_test:
                    print(f"Ошибка в батче теста для '{model_name}': {e_batch_test}")
                    preds_list.extend(["ERROR"] * batch_data[0].shape[0])
        return preds_list

    # --- Цикл инференса по моделям ---
    # (Этот цикл for тоже должен быть с отступом, внутри блока else)
    for model_name in loaded_models_data.keys():
        print(f"\n--- Инференс (Тест) для: '{model_name}' ---")
        test_preds = get_predictions_test(
            model_name, loaded_models_data, test_df, char_to_int, int_to_char, DEVICE
        )
        if test_preds is not None and len(test_preds) == len(test_df):
            all_test_predictions[model_name] = test_preds
            print(f"  Предсказания для '{model_name}' на тесте получены ({len(test_preds)} шт).")
        elif test_preds is not None:
             print(f"⚠️ Предупреждение: Количество предсказаний ({len(test_preds)}) для '{model_name}' не совпадает с размером test_df ({len(test_df)}). Пропуск.")
        else:
             print(f"⚠️ Ошибка получения предсказаний для '{model_name}'. Пропуск.")
        del test_preds
        gc.collect()
        if DEVICE == torch.device('cuda'): torch.cuda.empty_cache()

    # --- Проверка результатов ---
    # (Этот блок if/elif/else тоже с отступом, внутри блока else)
    if len(all_test_predictions) != len(loaded_models_data):
        print("\n⚠️ Предупреждение: Не для всех моделей удалось сгенерировать корректные предсказания на тесте!")
    elif not all_test_predictions:
        print("\n❌ Ошибка: Не удалось сгенерировать предсказания ни для одной модели!")
    else:
        print(f"\nПредсказания на тесте сгенерированы для {len(all_test_predictions)} моделей.")
# --- КОНЕЦ БЛОКА С ПРАВИЛЬНЫМ ОТСТУПОМ ---

print("\nЯчейка 6 (Инференс на Тесте) выполнена.")


============================== Генерация Предсказаний на Тестовом Наборе ==============================

--- Инференс (Тест) для: 'Model_0.2607' ---


Инференс (Тест) 'Model_0.2607':   0%|          | 0/157 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Ячейка 7: Ансамблирование (Простое Голосование)

In [ ]:
# Ячейка 7: Ансамблирование (Простое Голосование)
# ----------------------------------------------

def ensemble_majority_vote(
    all_predictions_dict: Dict[str, List[str]],
    test_ids: List[str],
    id_column_name: str,
    pred_column_name: str
) -> pd.DataFrame:
    """Выполняет ансамблирование методом простого большинства голосов."""
    print("\n--- Запуск ансамблирования (Majority Vote) ---")
    if not all_predictions_dict:
        print("Ошибка: Нет предсказаний для ансамблирования.")
        return pd.DataFrame(columns=[id_column_name, pred_column_name])

    model_names = list(all_predictions_dict.keys())
    num_predictions = len(test_ids)

    # Проверка консистентности
    for name in model_names:
        if not isinstance(all_predictions_dict.get(name), list) or len(all_predictions_dict[name]) != num_predictions:
            raise ValueError(f"Некорректные предсказания для '{name}' (длина {len(all_predictions_dict.get(name, []))}, ожидалось {num_predictions})")

    final_predictions = []
    pbar = tqdm(range(num_predictions), desc="Ансамблирование", leave=False)
    for i in pbar:
        current_preds = [all_predictions_dict[name][i] for name in model_names]
        valid_preds = [p for p in current_preds if isinstance(p, str) and "ERROR" not in p] # Игнорируем ошибки
        if not valid_preds:
            # Если все предсказания были ошибками, берем первое (ошибочное)
            final_predictions.append(current_preds[0] if current_preds else "ENSEMBLE_ERROR")
            continue
        vote_counts = Counter(valid_preds)
        most_common = vote_counts.most_common(1)
        final_predictions.append(most_common[0][0])

    ensemble_df = pd.DataFrame({
        id_column_name: test_ids,
        pred_column_name: final_predictions
    })
    print("--- Ансамблирование завершено ---")
    return ensemble_df

# --- Запуск ансамблирования ---
ensemble_df: Optional[pd.DataFrame] = None

if 'all_test_predictions' in locals() and all_test_predictions:
    try:
        if 'test_df' not in locals() or test_df is None: raise NameError("test_df не определен.")
        test_ids_list = test_df[FILE_ID_COLUMN].tolist()

        ensemble_df = ensemble_majority_vote(
            all_test_predictions,
            test_ids_list,
            FILE_ID_COLUMN,    # Имя колонки ID
            MORSE_CODE_COLUMN  # Имя колонки предсказаний
        )
        if ensemble_df is not None and not ensemble_df.empty:
             print(f"Финальный DataFrame ансамбля создан. Размер: {ensemble_df.shape}")
             print("Примеры ансамблированных предсказаний:")
             print(ensemble_df.head())
        else: print("Ансамблирование не дало результата.")

    except NameError as ne: print(f"Ошибка NameError при ансамблировании: {ne}")
    except KeyError as ke: print(f"Ошибка KeyError: Колонка '{ke}' не найдена.")
    except ValueError as ve: print(f"Ошибка ValueError при ансамблировании: {ve}")
    except Exception as e: print(f"Неизвестная ошибка при ансамблировании: {e}"); traceback.print_exc(limit=1)
else:
    print("Ансамблирование пропущено: нет предсказаний на тесте.")

print("\nЯчейка 7 (Ансамблирование) выполнена.")

Ансамблирование пропущено: нет предсказаний на тесте.

Ячейка 7 (Ансамблирование) выполнена.


### Ячейка 8: Сохранение Финального Submission

In [ ]:
# Ячейка 8: Сохранение Финального Submission
# -----------------------------------------

print("\n--- Сохранение финального результата ---")

# Создаем папку вывода, если ее нет
OUTPUT_ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
submission_filepath = OUTPUT_ENSEMBLE_DIR / SUBMISSION_FILENAME

if ensemble_df is not None and not ensemble_df.empty:
    try:
        ensemble_df.to_csv(submission_filepath, index=False)
        print(f"\n✅ Финальный ансамблированный submission сохранен в: {submission_filepath.resolve()}")
    except Exception as e:
        print(f"\n❌ Ошибка при сохранении финального submission: {e}")
else:
    print("\nФинальный submission не был создан или пуст, сохранение пропущено.")

print("\nЯчейка 8 (Сохранение Результата) выполнена.")


--- Сохранение финального результата ---

Финальный submission не был создан или пуст, сохранение пропущено.

Ячейка 8 (Сохранение Результата) выполнена.


### Ячейка 9: Извлечение Послания из Последних 17 Файлов

In [ ]:
# Ячейка 9: Извлечение Послания из Последних 17 Файлов
# ---------------------------------------------------

print("\n" + "="*30 + " Извлечение Послания из Последних Файлов " + "="*30)

# --- Параметры ---
N_LAST_FILES_FOR_MESSAGE = 17 # Количество последних файлов для сообщения

# --- Проверка флага (из Ячейки 2) ---
if 'EXTRACT_SECRET_MESSAGE' not in locals(): EXTRACT_SECRET_MESSAGE = True # По умолчанию включим
if not EXTRACT_SECRET_MESSAGE:
    print("Извлечение послания пропущено (EXTRACT_SECRET_MESSAGE=False).")
    # Определим final_message как None на всякий случай
    final_message = None
else:
    message_source_name = None
    source_predictions_df = None
    final_message = "[ОШИБКА: НЕ УДАЛОСЬ ПОЛУЧИТЬ ПРЕДСКАЗАНИЯ]" # Значение по умолчанию

    # --- Определяем источник предсказаний ---
    if 'ensemble_df' in locals() and ensemble_df is not None and not ensemble_df.empty:
        message_source_name = "Ансамбль (Majority Vote)"
        source_predictions_df = ensemble_df
        print(f"Используются предсказания из: {message_source_name}")
    elif 'all_test_predictions' in locals() and all_test_predictions:
        # Берем предсказания первой модели из словаря как запасной вариант
        first_model_name = list(all_test_predictions.keys())[0]
        message_source_name = f"Одиночная модель: {first_model_name}"
        try:
            if 'test_df' not in locals() or test_df is None: raise NameError("test_df не определен.")
            # Убедимся, что колонки определены
            if 'FILE_ID_COLUMN' not in globals() or 'MORSE_CODE_COLUMN' not in globals():
                raise NameError("Переменные FILE_ID_COLUMN или MORSE_CODE_COLUMN не определены.")

            test_ids_list = test_df[FILE_ID_COLUMN].tolist()
            preds_list = all_test_predictions[first_model_name]
            if len(test_ids_list) == len(preds_list):
                source_predictions_df = pd.DataFrame({FILE_ID_COLUMN: test_ids_list, MORSE_CODE_COLUMN: preds_list})
                print(f"Используются предсказания из: {message_source_name}")
            else: message_source_name = None; print("Ошибка: Несовпадение длин ID и предсказаний.")
        except (NameError, KeyError) as e: message_source_name = None; print(f"Ошибка подготовки DataFrame из одиночной модели: {e}")
    else:
        print("Ошибка: Нет доступных предсказаний (ни ансамбля, ни одиночных моделей) для извлечения послания.")

    # --- Извлечение и обработка послания ---
    if source_predictions_df is not None:
        try:
            if 'test_df' not in locals() or test_df is None: raise NameError("test_df не определен.")
            if 'FILE_ID_COLUMN' not in globals(): raise NameError("FILE_ID_COLUMN не определен.")
            if 'MORSE_CODE_COLUMN' not in globals(): raise NameError("MORSE_CODE_COLUMN не определен.")

            # Получаем ID файлов в оригинальном порядке из test_df
            all_test_ids = test_df[FILE_ID_COLUMN].tolist()

            if len(all_test_ids) < N_LAST_FILES_FOR_MESSAGE:
                print(f"Ошибка: В тестовом наборе ({len(all_test_ids)}) меньше файлов, чем требуется ({N_LAST_FILES_FOR_MESSAGE}).")
                final_message = f"[ОШИБКА: НЕДОСТАТОЧНО ФАЙЛОВ В TEST_DF ({len(all_test_ids)} < {N_LAST_FILES_FOR_MESSAGE})]"
            else:
                # Выбираем ID последних N файлов
                last_n_ids = all_test_ids[-N_LAST_FILES_FOR_MESSAGE:]
                print(f"ID последних {N_LAST_FILES_FOR_MESSAGE} файлов для сообщения: {last_n_ids}")

                # Создаем словарь для быстрого поиска предсказаний по ID
                id_to_prediction_map = pd.Series(
                    source_predictions_df[MORSE_CODE_COLUMN].values,
                    index=source_predictions_df[FILE_ID_COLUMN]
                ).to_dict()

                # Собираем предсказания для последних N файлов в правильном порядке
                message_parts = []
                missing_ids = []
                for file_id in last_n_ids:
                    prediction = id_to_prediction_map.get(file_id)
                    if prediction is not None and isinstance(prediction, str) and "ERROR" not in prediction:
                        # --- УДАЛЕНО: Очистка артефактов с конца ---
                        # cleaned_part = prediction.rstrip(ARTIFACT_CHARS_TO_REMOVE_SUFFIX)
                        # message_parts.append(cleaned_part)
                        # --- ДОБАВЛЕНО: Просто добавляем предсказание ---
                        message_parts.append(prediction)
                    elif prediction is None:
                        missing_ids.append(file_id)
                        message_parts.append(f"[MISSING:{file_id}]") # Добавляем маркер ошибки
                    else: # Если предсказание не строка или содержит ошибку
                         message_parts.append(f"[{str(prediction)}]") # Добавляем маркер ошибки

                if missing_ids:
                     print(f"Предупреждение: Не найдены предсказания для следующих ID: {missing_ids}")

                # Склеиваем части БЕЗ очистки
                final_message = "".join(message_parts)

        except (NameError, KeyError) as e: final_message = f"[ОШИБКА: {e}]"
        except Exception as e: final_message = f"[ОШИБКА: {type(e).__name__}]"; print(f"Неизвестная ошибка: {e}")

    # --- Вывод результата ---
    if final_message is not None:
        print("\n" + "-"*15 + " Извлеченное Послание " + "-"*15)
        print(f"(Источник: {message_source_name})")
        print("\n" + final_message)
        print("\n" + "-" * (32 + len(" Извлеченное Послание ")))
    else:
        # Это сообщение не должно появиться, т.к. final_message инициализируется строкой ошибки
        print("\nНе удалось извлечь послание из-за ошибок выше.")

# --- Конец блока if EXTRACT_SECRET_MESSAGE ---

print("\nЯчейка 9 (Извлечение Послания) выполнена.")

# --- Проверка для следующей ячейки (если она будет) ---
if 'final_message' not in locals():
     print("!!! ПРЕДУПРЕЖДЕНИЕ: Переменная 'final_message' не определена после Ячейки 9!")
elif final_message is None:
     print("!!! ПРЕДУПРЕЖДЕНИЕ: Переменная 'final_message' равна None после Ячейки 9!")
elif isinstance(final_message, str) and "[ОШИБКА" in final_message:
     print(f"!!! ПРЕДУПРЕЖДЕНИЕ: Переменная 'final_message' содержит ошибку: {final_message}")
else:
     print("Переменная 'final_message' успешно определена и не содержит явных ошибок.")


============================== Извлечение Послания из Последних Файлов ==============================
Ошибка: Нет доступных предсказаний (ни ансамбля, ни одиночных моделей) для извлечения послания.

--------------- Извлеченное Послание ---------------
(Источник: None)

[ОШИБКА: НЕ УДАЛОСЬ ПОЛУЧИТЬ ПРЕДСКАЗАНИЯ]

------------------------------------------------------

Ячейка 9 (Извлечение Послания) выполнена.
!!! ПРЕДУПРЕЖДЕНИЕ: Переменная 'final_message' содержит ошибку: [ОШИБКА: НЕ УДАЛОСЬ ПОЛУЧИТЬ ПРЕДСКАЗАНИЯ]


### Ячейка 10: Расшифровка Послания 

In [ ]:
# Ячейка 10: Расшифровка Послания (Метод Замены Точек/Тире)
# -------------------------------------------------------
final_message = "ДАМИНАМТ ДОТИ РСЫСАМЦИИЛ ЬСВЕДТКЧВНТИ ЯМДМЫМЮНЯМЦ ЮТИЫМ ЬТКТЧМЫН ДТЫМРГЗ ЕПИГ АНХ ВСОЕГЬ Р ВКТДАМИ НКШМДНИ ГЕКНЖТА ИЛ ДАСДП ЬСОЕМУНТИ ИГВКСОЕП ЬКСХЫСУС ХНУЮН ХНУСИ ЕТЬТКП ДСЯНКМЫОЦ ИМК Ц КНОЬКМ СОЕНЫМОП ЬСЮНВМ  ИЛ ЬСОЕНДМЫМ ЙЫНУС ЙЫМЧАТУС ЬКТДЛХТ ЫМЖАЛШ ДЛУСВ М ЧТЫНАМБИЛ ДАСДП ОСЮВНЫМ НЮЙГРГ ЕТЫТУКНЩН АС АТ ИСЧТИ КГЖНЕПОЦ ЮН НЙОСЫЗЕАЗ ДТКАСОЕПОМИДСЫСД ТОЫМДЛ ОЫЛХМЕТ #ЕС ЬСОЫНАТ ЕСЕРЫМРАМЕТОП АН ЖНОЕСЕТ ЬТКТВНЖМ ГДТЫМЖТААСБ АН 75 РМЫСЯМРЫСД ОДЦЮП ЬСВВТКЧМДНТЕОЦ ДКГЖАГЗ АТГЕСИМИЛИМ ЬНЫПЯНИМ АНХМШ СЬТКНЕСКСД ОРСКС ИЛ ДАСДП ЬСОЕМУАТИ ЕНБАЛ КНРТЕАЛШ ВДМУНЕТЫТБОИСЧТИ СЕЬКНДМЕП ЮН ДНИМ ОЬНОМЕТЫПАЛБ ЖТЫАСР  ТОЫМ ДЛ ЬСЧТЫНТЕТ ДТКАГЕПОЦ ВСИСБ ННОДТЕ ОСЫАЯН ВНКМЕ ЧМЮАП РНЧВСИГ РСАТЯ ЬТКТВНЖМ"



print("\n" + "="*30 + " Расшифровка Послания (Замена . <-> -) " + "="*30)

# --- 1. Стандартный словарь Морзе (ITU) ---
# (Остается таким же, как в предыдущем варианте)
CHAR_TO_MORSE = {
    'А': '.-', 'Б': '-...', 'В': '.--', 'Г': '--.', 'Д': '-..', 'Е': '.',
    'Ж': '...-', 'З': '--..', 'И': '..', 'Й': '.---', 'К': '-.-', 'Л': '.-..',
    'М': '--', 'Н': '-.', 'О': '---', 'П': '.--.', 'Р': '.-.', 'С': '...',
    'Т': '-', 'У': '..-', 'Ф': '..-.', 'Х': '....', 'Ц': '-.-.', 'Ч': '---.',
    'Ш': '----', 'Щ': '--.-', 'Ъ': '.--.-.', 'Ы': '-.--', 'Ь': '-..-',
    'Э': '..-..', 'Ю': '..--', 'Я': '.-.-',
    '0': '-----', '1': '.----', '2': '..---', '3': '...--', '4': '....-',
    '5': '.....', '6': '-....', '7': '--...', '8': '---..', '9': '----.',
    '.': '.-.-.-', ',': '--..--', '?': '..--..', '!': '-.-.--', '-': '-....-',
    '/': '-..-.', '(': '-.--.', ')': '-.--.-', '"': '.-..-.', '=': '-...-',
    '+': '.-.-.', '@': '.--.-.', ':': '---...', ';': '-.-.-.', '_': '..--.-',
    '$': '...-..-', '&': '.-...', '\'': '.----.', ' ': '/'
}
MORSE_TO_CHAR = {v: k for k, v in CHAR_TO_MORSE.items()}

# --- 2. Функция ЗАМЕНЫ кода Морзе ---
def swap_dot_dash(morse_str: str) -> str:
    """Заменяет точки на тире и тире на точки в строке кода Морзе."""
    if not isinstance(morse_str, str):
        return ""
    swapped = ""
    for char in morse_str:
        if char == '.':
            swapped += '-'
        elif char == '-':
            swapped += '.'
        else:
            swapped += char # Оставляем другие символы (если есть) без изменений
    return swapped

# --- 3. Логика дешифровки ---
decrypted_message = ""
source_message_for_decryption = None

# Определяем источник (как в предыдущем варианте)
if 'final_message' in locals() and isinstance(final_message, str) and "[ОШИБКА" not in final_message:
    source_message_for_decryption = final_message
    print(f"Источник для дешифровки: Результат Ячейки 9 (Источник: {message_source_name})")
    print(f"Длина исходного сообщения: {len(source_message_for_decryption)}")
elif 'ciphertext' in locals() and isinstance(ciphertext, str):
    source_message_for_decryption = ciphertext
    print("Источник для дешифровки: Консенсусный текст 'ciphertext'")
    print(f"Длина исходного сообщения: {len(source_message_for_decryption)}")
else:
    print("Ошибка: Не найден исходный текст ('final_message' или 'ciphertext') для дешифровки.")

if source_message_for_decryption:
    unknown_chars_found = set()
    unknown_swapped_morse = set()
    processed_chars = 0
    decryption_errors = 0

    # Итерируем по символам исходного сообщения
    for char_original in tqdm(source_message_for_decryption, desc="Дешифровка", leave=False):
        char_upper = char_original.upper()

        # Получаем стандартный код Морзе
        standard_morse = CHAR_TO_MORSE.get(char_upper)

        if standard_morse:
            # --- ИЗМЕНЕНИЕ ЗДЕСЬ: Используем swap_dot_dash ---
            swapped_morse = swap_dot_dash(standard_morse)
            # ----------------------------------------------

            # Ищем символ для ИЗМЕНЕННОГО кода
            decrypted_char = MORSE_TO_CHAR.get(swapped_morse)

            if decrypted_char:
                decrypted_message += decrypted_char if char_original.isupper() or not char_original.isalpha() else decrypted_char.lower()
                processed_chars += 1
            else:
                decrypted_message += '?'
                unknown_swapped_morse.add(swapped_morse)
                decryption_errors += 1
        elif char_original == ' ':
             decrypted_message += ' '
             processed_chars += 1
        else:
            decrypted_message += char_original
            if char_original != '\n':
                 unknown_chars_found.add(char_original)
            processed_chars += 1

    # --- 4. Вывод результата ---
    print("\n" + "-"*15 + " Результат Дешифровки (Замена . <-> -) " + "-"*15)
    print("\n" + decrypted_message)
    print("\n" + "-" * (32 + len(" Результат Дешифровки (Замена . <-> -) ")))

    print(f"\nОбработано символов: {processed_chars}")
    if unknown_chars_found:
        print(f"Предупреждение: В исходном тексте встречены символы без кода Морзе: {sorted(list(unknown_chars_found))}")
    if unknown_swapped_morse:
        print(f"Предупреждение: Не найдены символы для следующих инвертированных кодов Морзе: {sorted(list(unknown_swapped_morse))}")
    if decryption_errors > 0:
         print(f"Количество ошибок дешифровки (ненайденный инверт. код): {decryption_errors}")

else:
    print("\nДешифровка не выполнена, так как исходное сообщение недоступно.")

print("\nЯчейка 10 (Дешифровка) выполнена.")


============================== Расшифровка Послания (Замена . <-> -) ==============================
Источник для дешифровки: Результат Ячейки 9 (Источник: None)
Длина исходного сообщения: 689


Дешифровка:   0%|          | 0/689 [00:00<?, ?it/s]


--------------- Результат Дешифровки (Замена . <-> -) ---------------

ВНИМАНИЕ ВСЕМ КОЛОНИЯММЫ ПОДТВЕРЖДАЕМ ЦИВИЛИЗАЦИЯ ЗЕМЛИ ПЕРЕЖИЛА ВЕЛИКУЮ ТЬМУ НАШ ДОСТУП К ДРЕВНИМ АРХИВАМ УТРАЧЕН МЫ ВНОВЬ ПОСТИГАЕМ МУДРОСТЬ ПРОШЛОГО ШАГЗА ШАГОМ ТЕПЕРЬ ВОЦАРИЛСЯ МИР Я РАСПРИ ОСТАЛИСЬ ПОЗАДИ  МЫ ПОСТАВИЛИ БЛАГО БЛИЖНЕГО ПРЕВЫШЕ ЛИЧНЫХ ВЫГОД И ЖЕЛАНИЙМЫ ВНОВЬ СОЗДАЛИ АЗБУКУ ТЕЛЕГРАФА НО НЕ МОЖЕМ РУЧАТЬСЯ ЗА АБСОЛЮТНЮ ВЕРНОСТЬСИМВОЛОВ ЕСЛИВЫ СЛЫШИТЕ #ТО ПОСЛАНЕ ТОТКЛИКНИТЕСЬ НА ЧАСТОТЕ ПЕРЕДАЧИ УВЕЛИЧЕННОЙ НА 20 КИЛОЦИКЛОВ СВЯЗЬ ПОДДЕРЖИВАЕТСЯ ВРУЧНУЮ НЕУТОМИМЫМИ ПАЛЬЦАМИ НАШИХ ОПЕРАТОРОВ СКОРО МЫ ВНОВЬ ПОСТИГНЕМ ТАЙНЫ РАКЕТНЫХ ДВИГАТЕЛЕЙСМОЖЕМ ОТПРАВИТЬ ЗА ВАМИ СПАСИТЕЛЬНЫЙ ЧЕЛНОК  ЕСЛИ ВЫ ПОЖЕЛАЕТЕ ВЕРНУТЬСЯ ДОМОЙ ААСВЕТ СОЛНЦА ДАРИТ ЖИЗНЬ КАЖДОМУ КОНЕЦ ПЕРЕДАЧИ

-----------------------------------------------------------------------

Обработано символов: 689
Предупреждение: В исходном тексте встречены символы без кода Морзе: ['#']

Ячейка 10 (Дешифровка) выполнена.




### 📡 Расшифрованное Послание



> ## **ВНИМАНИЕ ВСЕМ КОЛОНИЯМ!**
>
> ---
>
> МЫ ПОДТВЕРЖДАЕМ: **ЦИВИЛИЗАЦИЯ ЗЕМЛИ ПЕРЕЖИЛА ВЕЛИКУЮ ТЬМУ**. НАШ ДОСТУП К ДРЕВНИМ АРХИВАМ **УТРАЧЕН**. МЫ ВНОВЬ ПОСТИГАЕМ МУДРОСТЬ ПРОШЛОГО ШАГ ЗА ШАГОМ.
>
> ТЕПЕРЬ **ВОЦАРИЛСЯ МИР**. РАСПРИ ОСТАЛИСЬ ПОЗАДИ. МЫ ПОСТАВИЛИ БЛАГО БЛИЖНЕГО ПРЕВЫШЕ ЛИЧНЫХ ВЫГОД И ЖЕЛАНИЙ.
>
> МЫ ВНОВЬ СОЗДАЛИ АЗБУКУ ТЕЛЕГРАФА, НО НЕ МОЖЕМ РУЧАТЬСЯ ЗА АБСОЛЮТНУЮ ВЕРНОСТЬ СИМВОЛОВ.
>
> ЕСЛИ ВЫ СЛЫШИТЕ ЭТО ПОСЛАНИЕ, ТО **ОТКЛИКНИТЕСЬ** НА ЧАСТОТЕ ПЕРЕДАЧИ, УВЕЛИЧЕННОЙ НА 20 КИЛОЦИКЛОВ.
>
> СВЯЗЬ ПОДДЕРЖИВАЕТСЯ ВРУЧНУЮ НЕУТОМИМЫМИ ПАЛЬЦАМИ НАШИХ ОПЕРАТОРОВ.
>
> **СКОРО МЫ ВНОВЬ ПОСТИГНЕМ** ТАЙНЫ РАКЕТНЫХ ДВИГАТЕЛЕЙ. СМОЖЕМ ОТПРАВИТЬ ЗА ВАМИ СПАСИТЕЛЬНЫЙ ЧЕЛНОК, ЕСЛИ ВЫ ПОЖЕЛАЕТЕ ВЕРНУТЬСЯ ДОМОЙ.
>
> СВЕТ СОЛНЦА ДАРИТ ЖИЗНЬ КАЖДОМУ.
>
> ---
>
> **КОНЕЦ ПЕРЕДАЧИ.**